In [1]:
from docling.document_converter import DocumentConverter

converter = DocumentConverter()
result = converter.convert("../data/text_files/Reliance.pdf")
doc = result.document

/Users/ishitarastogi/Documents/langchain-rag-pipeline/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[INFO] 2026-08-16 15:14:40,526 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-16 15:14:40,536 [RapidOCR] download_file.py:60: File exists and is valid: /Users/ishitarastogi/Documents/langchain-rag-pipeline/.venv/lib/python3.11/site-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-08-16 15:14:40,537 [RapidOCR] main.py:63: Using /Users/ishitarastogi/Documents/langchain-rag-pipeline/.venv/lib/python3.11/site-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-08-16 15:14:40,618 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-16 15:14:40,620 [RapidOCR] download_file.py:60: File exists and is valid: /Users/ishitarastogi/Documents/langchain

In [2]:
print(type(doc))
print(len(doc.pages))
print(len(doc.tables))

<class 'docling_core.types.doc.document.DoclingDocument'>
174
385


In [3]:
indices = [0, len(doc.tables) // 2, len(doc.tables) - 1]

for i in indices:
    table = doc.tables[i]
    page = table.prov[0].page_no if table.prov else "unknown"
    print(f"=== Table {i} (page {page}) ===")
    print(table.export_to_markdown())
    print()

Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.


=== Table 0 (page 1) ===
| Notice                   | https://www.ril.com/sites/default/files/reports/Notice-of- 48th-Annual-General-Meeting-Post-IPO.pdf   |
|--------------------------|-------------------------------------------------------------------------------------------------------|
| Integrated Annual Report | https://www.ril.com/sites/default/files/reports/RIL_IAR_ 2025.pdf                                     |

=== Table 192 (page 108) ===
|  Particulars                                                                                     |         | ( C in crore)   |
|-------------------------------------------------------------------------------------|---------|-----------------|
|                                                                                     | 2024-25 | 2023-24         |
| Reliance Chemicals and Materials Limited                                            | 3       | -               |
| Reliance Corporate IT Park Limited                                

In [4]:
# Check 1: confirm page numbering convention (0-indexed vs 1-indexed)
for i, table in enumerate(doc.tables[:5]):
    page = table.prov[0].page_no if table.prov else "?"
    print(f"Table {i}: page {page}")

print()

# Check 2: preview text from Docling's markdown export, page by page
markdown = doc.export_to_markdown()
print(f"Total markdown length: {len(markdown)} characters")
print()

# Peek at the raw markdown for the first chunk of content
print(markdown[:2000])

Table 0: page 1
Table 1: page 8
Table 2: page 8
Table 3: page 11
Table 4: page 12

Total markdown length: 1999914 characters

BSE Limited Phiroze Jeejeebhoy Towers, Dalal Street, Mumbai 400 001

Scrip Code:

500325

Dear Sirs,

Sub

: Notice  of  the  Forty-eighth  Annual  General  Meeting  (Post-IPO)  and  the Integrated Annual Report for the financial year 2024-25

Notice convening the Forty-eighth Annual General Meeting (Post-IPO) ('Notice') and the Integrated Annual Report of the Company, for the financial year 2024-25, are being sent through electronic mode to all the members and debenture holders whose  e-mail  address  is  registered  with  the  Company /  Company's  Registrar  and Transfer Agent / Depository Participants / Depositories.

Notice and Integrated Annual Report are attached and the same are also available on the Company's website at:

| Notice                   | https://www.ril.com/sites/default/files/reports/Notice-of- 48th-Annual-General-Meeting-Post-IPO.pdf   |


In [5]:
from collections import defaultdict

# Group text items by page
page_texts = defaultdict(list)

for item, _level in doc.iterate_items():
    text = getattr(item, "text", None)
    if not text:
        continue
    if not getattr(item, "prov", None):
        continue
    page = item.prov[0].page_no
    page_texts[page].append(text)

# Preview first 30 pages
for page_num in sorted(page_texts.keys())[:30]:
    combined = " ".join(page_texts[page_num])
    preview = combined[:100].replace("\n", " ")
    print(f"{page_num:3d} | {len(combined):5d} | {preview}")

  1 |  1504 | BSE Limited Phiroze Jeejeebhoy Towers, Dalal Street, Mumbai 400 001 Scrip Code: 500325 Dear Sirs, Su
  2 |   956 | Copy to: Luxembourg Stock Exchange Singapore Exchange Limited 35A Boulevard Joseph II, L-1840 Luxemb
  3 |  4735 | Registered Office : 3 rd CIN : L17110MH1973PLC019786 Floor, Maker Chambers IV, 222, Nariman Point, M
  4 |  5153 |  To approve Material Related Party Transactions of the Company and in this regard, to consider and 
  5 |  5407 | Notes:  The Ministry of Corporate Affairs (' MCA ') has, vide its General Circular dated September 
  6 |  4723 | Procedure for joining the AGM through VC / OAVM:  The Company will provide VC / OAVM facility to it
  7 |  4979 |  The remote e-voting will not be allowed beyond the aforesaid date and time and the remote e-voting
  8 |  1015 | Procedure to login through their demat accounts / Website of Depository Participant Individual membe
  9 |  4568 | (vii)  INFORMATION AND INSTRUCTIONS FOR REMOTE E-VOTING BY (I) ME

In [6]:
CONTENT_START = 28

text_records = []

for page_num in sorted(page_texts.keys()):
    if page_num < CONTENT_START:
        continue

    combined = " ".join(page_texts[page_num])

    if len(combined.split()) < 30:  # skip near-empty pages
        continue

    text_records.append({
        "text": combined,
        "company": "Reliance",
        "fiscal_year": "2024-25",
        "page": page_num,
        "type": "text"
    })

print(f"{len(text_records)} text records")
print(text_records[0])

137 text records
{'text': "Integrated Annual Report 2024-25 Integrated Annual Report Reliance Industries Limited (RIL) is India's largest private sector enterprise and a Fortune Global 500 leader. Its presence across energy, retail, telecom, media and green technologies touches millions of lives every day, contributing to the nation's unstoppable growth momentum. As India dreams bigger and strides confidently forward, Reliance fuels that ambition with relentless passion, living by its unshakeable belief that 'Growth is Life'.", 'company': 'Reliance', 'fiscal_year': '2024-25', 'page': 28, 'type': 'text'}


In [7]:
table_records = []

for i, table in enumerate(doc.tables):
    if not table.prov:
        continue

    page_num = table.prov[0].page_no

    if page_num < CONTENT_START:
        continue

    # doc arg avoids the deprecation warning
    markdown = table.export_to_markdown(doc)

    if len(markdown.strip()) < 40:  # skip near-empty tables
        continue

    table_records.append({
        "text": markdown,
        "company": "Reliance",
        "fiscal_year": "2024-25",
        "page": page_num,
        "type": "table",
        "table_index": i
    })

print(f"{len(table_records)} table records")
print(table_records[0])

365 table records
{'text': '|                                                                       | US$ million   | FY 2024-25   | FY 2023-24   | FY 2022-23***   | FY 2021-22   | FY 2020-21   | FY 2019-20   | FY 2018-19   | FY 2017-18   | FY 2016-17   | FY 2015-16   |\n|-----------------------------------------------------------------------|---------------|--------------|--------------|-----------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|\n| Value of Sales and Services (Revenue)                                 | 125,320       | 10,71,174    | 10,00,122    | 9,74,864        | 7,88,743     | 5,39,238     | 6,59,997     | 6,25,212     | 4,30,731     | 3,30,180     | 2,93,298     |\n| Total Income                                                          | 116,773       | 9,98,114     | 9,30,529     | 9,03,045        | 7,32,578     | 5,02,653     | 6,25,601     | 5,91,480     | 4,18,214     | 3,39,623     | 3,05,351    

In [8]:
import json

all_records = text_records + table_records

print(f"Total records: {len(all_records)}")

with open("../data/reliance_records.json", "w", encoding="utf-8") as f:
    json.dump(all_records, f, indent=2, ensure_ascii=False)

print("Saved.")

Total records: 502
Saved.


In [9]:
def chunk_text(text, size=1000, overlap=200):
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start:start + size])
        start += size - overlap
    return chunks


all_chunks = []

for record in all_records:
    if record["type"] == "text":
        pieces = chunk_text(record["text"])
        for i, piece in enumerate(pieces):
            all_chunks.append({
                "text": piece,
                "company": record["company"],
                "fiscal_year": record["fiscal_year"],
                "page": record["page"],
                "type": "text",
                "chunk_index": i
            })
    elif record["type"] == "table":
        all_chunks.append({
            "text": record["text"],
            "company": record["company"],
            "fiscal_year": record["fiscal_year"],
            "page": record["page"],
            "type": "table",
            "table_index": record.get("table_index")
        })

print(f"{len(all_chunks)} total chunks")

1154 total chunks


In [10]:
with open("../data/reliance_chunks.json", "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, indent=2, ensure_ascii=False)

print("Saved.")

# Spot-check: one text chunk, one table chunk
text_chunks = [c for c in all_chunks if c["type"] == "text"]
table_chunks = [c for c in all_chunks if c["type"] == "table"]

print(f"\n{len(text_chunks)} text chunks, {len(table_chunks)} table chunks\n")

print("--- Sample text chunk ---")
print(text_chunks[10]["text"][:400])

print("\n--- Sample table chunk ---")
print(table_chunks[5]["text"][:400])

Saved.

789 text chunks, 365 table chunks

--- Sample text chunk ---
What is Good for India is Good for Reliance Dear Shareholders, Nearly fifty years ago, our visionary founder, Shri Dhirubhai Ambani, embarked on a bold mission to prove that India could build a world-class enterprise founded on innovation, integrity, and ambition. That belief became Reliance. And over the decades, Reliance has grown from an idea into one of the world's most admired enterprises a s

--- Sample table chunk ---
|                             | FY 2024-25 *   | FY 2023-24   | Y-o-Y Change   |
|-----------------------------|----------------|--------------|----------------|
| Value of sales and services | 20,696         | 11,875       | 74.3%          |
| Revenue from operations     | 17,762         | 10,157       | 74.9%          |
| EBITDA                      | 1,833          | 765          | 139.6%      
